# 7. Customer Enrichment

Synthetic client dimension table (FOC-180, phase F1, chunk A).

The transaction data (`all_trxns.csv`) carries no client attributes - only the
anonymous `customer` id. To give the model customer-level signal we generate a
synthetic `dim_customer` table with realistic demographic and behavioural
attributes:

- `gender`, `age` - demographics
- `employment_industry`, `employment_status` - employment
- `account_tenure_years` - relationship length (correlated with age)
- `income_band` - banded disposable income (lognormal)
- `device`, `channel` - preferred interaction hardware / channel

Design constraints:

- **No PII (phase decision D3)** - attributes are fully synthetic; no names,
  emails or account numbers.
- **Deterministic** - every customer's attributes are seeded by hashing their
  `customer` id (SHA-256), not by a global RNG seed. Re-running the notebook
  reproduces the exact same table, on any machine.
- **Few categories** - every categorical column has <= 8 levels so one-hot
  encoding stays compact on 100 unique customers.


In [1]:
import hashlib
import random

import pandas as pd

from funs import dataPreparation

## Source of customer ids

The `customer` column of the raw transactions is the only join key we have.
Each unique id maps to exactly one synthetic profile row.


In [2]:
all_trxns = pd.read_csv("../data/all_trxns.csv", dtype={"customer": str})
customer_ids = sorted(all_trxns["customer"].unique())

print("transactions:", len(all_trxns))
print("unique customers:", len(customer_ids))
print("sample ids:", customer_ids[:3])


transactions: 5302
unique customers: 100
sample ids: ['C12976926337644', 'C14368611669296', 'C17694553858863']


## Deterministic profile generation

Each attribute draw comes from a `random.Random` instance seeded with
`SHA-256(customer_id)` - an integer derived from the id itself, so the seed
travels with the customer:

- same id -> same seed -> same profile, on every run and every machine;
- no global seed to lose or mismatch between notebooks;
- the per-customer draw order is fixed (documented in `build_customer_profile`),
  which keeps the id -> profile mapping stable if the table is regenerated.

`generate_dim_customer()` is a pure function of the customer id list, so we
call it twice in this run and assert the two frames are identical - an
explicit regression guard for the determinism contract.


In [3]:
# Ordinal income bands: label prefixes keep sort order == band order.
INCOME_BANDS = [
    (20_000, "1_<20k"),
    (40_000, "2_20k_40k"),
    (60_000, "3_40k_60k"),
    (90_000, "4_60k_90k"),
    (140_000, "5_90k_140k"),
    (float("inf"), "6_>140k"),
]

DIM_CUSTOMER_COLUMNS = [
    "customer",
    "gender",
    "age",
    "employment_industry",
    "employment_status",
    "account_tenure_years",
    "income_band",
    "device",
    "channel",
]


def hash_seed(customer_id):
    # SHA-256 of the id -> big int; deterministic across runs and machines.
    digest = hashlib.sha256(customer_id.encode("utf-8")).digest()
    return int.from_bytes(digest, "big")


def build_customer_profile(customer_id):
    # One profile per customer, drawn from a per-customer seeded RNG.
    # Draw order is fixed: gender, age, status, industry, tenure, income,
    # device, channel. All randomness is stdlib `random` - no global RNG state.
    rng = random.Random(hash_seed(customer_id))

    # Age ~ normal, clipped to a plausible banking-client range.
    age = int(min(85, max(18, round(rng.gauss(45, 15)))))

    gender = rng.choices(["F", "M"], weights=[51, 49])[0]

    # Employment status follows age (students young, retirees old).
    if age >= 65:
        employment_status = rng.choices(
            ["retired", "self_employed", "employed"], weights=[80, 10, 10]
        )[0]
    elif age < 25:
        employment_status = rng.choices(
            ["student", "employed", "unemployed"], weights=[45, 40, 15]
        )[0]
    else:
        employment_status = rng.choices(
            ["employed", "self_employed", "unemployed"], weights=[70, 15, 15]
        )[0]

    employment_industry = rng.choices(
        [
            "technology",
            "finance",
            "retail",
            "healthcare",
            "education",
            "manufacturing",
            "construction",
            "public_sector",
        ],
        weights=[15, 15, 15, 12, 10, 12, 8, 13],
    )[0]

    # Tenure cannot exceed the customer's adult life, so young customers are
    # forced short automatically; older ones get a growing expected tenure.
    max_tenure = age - 18
    tenure = round(rng.gauss(0.35 * max_tenure + 2, 4))
    account_tenure_years = int(min(max(tenure, 0), max_tenure))

    # Income ~ lognormal, shifted up with career stage (plateaus at 55).
    career = min(max(age - 18, 0), 37) / 37.0
    income = rng.lognormvariate(9.5 + 1.3 * career, 0.45)
    income_band = next(label for ceiling, label in INCOME_BANDS if income < ceiling)

    # Mobile-first interaction skews.
    device = rng.choices(["mobile", "desktop", "tablet"], weights=[55, 30, 15])[0]
    channel = rng.choices(
        ["mobile_app", "online_banking", "branch", "phone"],
        weights=[50, 25, 15, 10],
    )[0]

    return {
        "customer": customer_id,
        "gender": gender,
        "age": age,
        "employment_industry": employment_industry,
        "employment_status": employment_status,
        "account_tenure_years": account_tenure_years,
        "income_band": income_band,
        "device": device,
        "channel": channel,
    }


def generate_dim_customer(customer_ids):
    # Pure function of the id list: seeds come from the ids themselves.
    profiles = [build_customer_profile(cid) for cid in customer_ids]
    return pd.DataFrame(profiles, columns=DIM_CUSTOMER_COLUMNS)


print("generator ready:", len(DIM_CUSTOMER_COLUMNS) - 1, "attributes per customer")


generator ready: 8 attributes per customer


In [4]:
dim_customer_run1 = generate_dim_customer(customer_ids)
dim_customer_run2 = generate_dim_customer(customer_ids)

# Determinism contract: two independent builds of the same id list must be
# identical - if not, this cell (and the notebook) fails.
determinism_ok = dim_customer_run1.equals(dim_customer_run2)
assert determinism_ok, "dim_customer generation is not deterministic"
print("determinism check: PASS - two independent builds are identical")


determinism check: PASS - two independent builds are identical


In [5]:
DIM_CUSTOMER_PATH = "../data/dim_customer.csv"
dim_customer = dim_customer_run1
dim_customer.to_csv(DIM_CUSTOMER_PATH, index=False)
print("written:", DIM_CUSTOMER_PATH, dim_customer.shape)
dim_customer.head()


written: ../data/dim_customer.csv (100, 9)


,customer,gender,age,employment_industry,employment_status,account_tenure_years,income_band,device,channel
0,C12976926337644,F,39,technology,unemployed,9,2_20k_40k,mobile,branch
1,C14368611669296,M,49,education,employed,10,3_40k_60k,desktop,mobile_app
2,C17694553858863,F,34,technology,employed,16,2_20k_40k,desktop,mobile_app
3,C24211332442813,M,19,technology,unemployed,1,1_<20k,mobile,mobile_app
4,C26191441143115,F,31,retail,self_employed,7,2_20k_40k,desktop,phone


## Distribution sanity checks

Quick look at the generated distributions: plausible age spread, income
right-skewed (lognormal, banded), mobile-first device/channel skews, tenure
growing with age. Every categorical stays within the <= 8 levels budget.


In [6]:
print(dim_customer[["age", "account_tenure_years"]].describe().round(2), "\n")

for col in [
    "gender",
    "employment_status",
    "employment_industry",
    "income_band",
    "device",
    "channel",
]:
    counts = dim_customer[col].value_counts()
    if col == "income_band":
        counts = counts.sort_index()  # ordinal bands in band order
    print(col, "(%d levels)" % len(counts))
    print(counts.to_string(), "\n")

corr = dim_customer["age"].corr(dim_customer["account_tenure_years"])
print("age vs tenure correlation: %.2f (should be clearly positive)" % corr)


          age  account_tenure_years
count  100.00                100.00
mean    44.33                 11.21
std     14.07                  6.03
min     18.00                  0.00
25%     33.75                  7.00
50%     44.00                 11.50
75%     54.00                 16.00
max     82.00                 25.00 

gender (2 levels)
gender
F    55
M    45 

employment_status (5 levels)
employment_status
employed         67
unemployed       15
self_employed    11
retired           5
student           2 

employment_industry (8 levels)
employment_industry
retail           24
manufacturing    14
public_sector    13
technology       12
healthcare       12
finance          11
education         8
construction      6 

income_band (5 levels)
income_band
1_<20k       26
2_20k_40k    44
3_40k_60k    20
4_60k_90k     9
6_>140k       1 

device (3 levels)
device
mobile     48
desktop    38
tablet     14 

channel (4 levels)
channel
mobile_app        51
online_banking    23
branch        

## Join with prepared transactions

Enrich the `dataPreparation()` output with the synthetic client attributes.
The merge is validated as `many_to_one` (many transactions per single
customer row), and we check that no rows are lost and no nulls are
introduced by the new columns.


In [7]:
trxns = dataPreparation(
    all_trxns_path="../data/all_trxns.csv",
    exchange_rates_path="../data/exchange_rates.csv",
)
new_cols = [c for c in DIM_CUSTOMER_COLUMNS if c != "customer"]

enriched = trxns.merge(dim_customer, on="customer", how="left", validate="many_to_one")

print("trxns shape:   ", trxns.shape)
print("enriched shape:", enriched.shape)

assert len(enriched) == len(trxns), "join lost rows"
assert enriched["customer"].nunique() == dim_customer["customer"].nunique(), (
    "join changed the customer set"
)
assert int(enriched[new_cols].isna().sum().sum()) == 0, "join introduced nulls"
print("sanity checks: PASS - no rows lost, no nulls introduced")
enriched[["customer", "fraud_flag"] + new_cols].head()


trxns shape:    (5302, 19)
enriched shape: (5302, 27)
sanity checks: PASS - no rows lost, no nulls introduced


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-180-f1\src\funs.py:197: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  trxns_data["timestamp"] = pd.to_datetime(


,customer,fraud_flag,gender,age,employment_industry,employment_status,account_tenure_years,income_band,device,channel
0,C12976926337644,N,F,39,technology,unemployed,9,2_20k_40k,mobile,branch
1,C12976926337644,N,F,39,technology,unemployed,9,2_20k_40k,mobile,branch
2,C12976926337644,N,F,39,technology,unemployed,9,2_20k_40k,mobile,branch
3,C12976926337644,N,F,39,technology,unemployed,9,2_20k_40k,mobile,branch
4,C12976926337644,N,F,39,technology,unemployed,9,2_20k_40k,mobile,branch


## Summary

- `data/dim_customer.csv` written: 100 rows (one per unique customer), 9
  columns, no PII.
- Generation is deterministic per customer id (SHA-256 seeding) and verified
  by the in-run double-build assertion.
- The enriched transaction frame carries the original features plus the 8 new
  client attributes. The A/B model evaluation - baseline vs enriched
  features - follows in the next chunk of this phase.
